# Model Training

This notebook trains the baseline fraud-detection models with a holdout test set and `StratifiedKFold` validation on the training split.


In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path("..").resolve()))

from src.config import PROJECT_ROOT, RANDOM_STATE, TARGET_COLUMN, TEST_SIZE
from src.features.feature_selection import get_selected_feature_names, load_feature_selection_decisions

NOTEBOOK_NAME = "13_model_training"
SELECTED_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "creditcard_selected_features.csv"
NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / NOTEBOOK_NAME
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / NOTEBOOK_NAME
NOTEBOOK_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / NOTEBOOK_NAME
N_SPLITS = 5

NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Modeling data  :", SELECTED_DATA_FILE)
print("Tables dir     :", NOTEBOOK_TABLES_DIR)
print("Artifacts dir  :", NOTEBOOK_ARTIFACTS_DIR)
print("K-fold splits  :", N_SPLITS)


## 1. Objectives

- Load the finalized selected-feature dataset from notebook `10_feature_selection`.
- Create one untouched holdout test split for final model evaluation.
- Apply `StratifiedKFold` only on the training split to compare baseline models fairly.
- Retrain each baseline model on the full training split after cross-validation.
- Save fitted models, cross-validation summaries, and holdout probabilities for notebook `14_model_evaluation`.

## Output Guide

This notebook writes its main outputs to:

- `reports/tables/13_model_training/`
- `artifacts/13_model_training/`


## 2. Load Final Modeling Data

The baseline models should use only the finalized selected features plus the target column.


In [ ]:
df = pd.read_csv(SELECTED_DATA_FILE)
selected_features = get_selected_feature_names(load_feature_selection_decisions())

expected_columns = selected_features + [TARGET_COLUMN]
missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Selected dataset is missing expected columns: {missing_columns}")

model_df = df.loc[:, expected_columns].copy()
model_df["transaction_id"] = np.arange(len(model_df))

print(f"Selected dataset shape: {model_df.shape}")
print(f"Selected feature count: {len(selected_features)}")
print(f"Fraud rate: {model_df[TARGET_COLUMN].mean():.6f}")
model_df.head()


## 3. Define Inputs and Target

The notebook keeps the modeling inputs explicit so downstream evaluation can trace exactly which features were used.


In [ ]:
X = model_df[selected_features].copy()
y = model_df[TARGET_COLUMN].copy()
row_ids = model_df["transaction_id"].copy()

input_summary = pd.DataFrame(
    [
        {"metric": "row_count", "value": int(len(model_df))},
        {"metric": "feature_count", "value": int(len(selected_features))},
        {"metric": "target_column", "value": TARGET_COLUMN},
        {"metric": "fraud_rate", "value": float(y.mean())},
    ]
)
input_summary.to_csv(NOTEBOOK_TABLES_DIR / "input_data_summary.csv", index=False)
input_summary


## 4. Holdout Train/Test Split

The test set is created once and kept untouched during baseline model comparison. All K-fold validation happens only inside the training split.


In [ ]:
X_train, X_test, y_train, y_test, row_id_train, row_id_test = train_test_split(
    X,
    y,
    row_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": int(len(X_train)),
            "fraud_count": int(y_train.sum()),
            "fraud_rate": float(y_train.mean()),
        },
        {
            "split": "test",
            "row_count": int(len(X_test)),
            "fraud_count": int(y_test.sum()),
            "fraud_rate": float(y_test.mean()),
        },
    ]
)
split_summary.to_csv(NOTEBOOK_TABLES_DIR / "train_test_split_summary.csv", index=False)
split_summary


## 5. Baseline Preprocessing and Validation Setup

- Logistic Regression uses imputation plus scaling because coefficient-based models are sensitive to feature scale.
- Random Forest keeps a lighter preprocessing path because tree-based models do not require scaling.
- `StratifiedKFold` is applied only to `X_train` and `y_train`.


In [ ]:
numeric_features = selected_features.copy()

logreg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        )
    ],
    remainder="drop",
)

cv_strategy = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

preprocessing_summary = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "preprocessing": "median_imputation + standard_scaling",
            "validation": f"StratifiedKFold(n_splits={N_SPLITS}) on training split",
        },
        {
            "model_name": "random_forest",
            "preprocessing": "median_imputation",
            "validation": f"StratifiedKFold(n_splits={N_SPLITS}) on training split",
        },
    ]
)
preprocessing_summary.to_csv(NOTEBOOK_TABLES_DIR / "preprocessing_summary.csv", index=False)
preprocessing_summary


## 6. Cross-Validation Helpers

The helper below trains each baseline model across the training folds, records fold-level metrics, and returns out-of-fold probabilities for later inspection.


In [ ]:
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def run_training_cv(model_name, pipeline, X_train, y_train, row_id_train, cv_strategy):
    fold_rows = []
    oof_rows = []

    for fold_number, (fit_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
        X_fit = X_train.iloc[fit_idx]
        X_valid = X_train.iloc[valid_idx]
        y_fit = y_train.iloc[fit_idx]
        y_valid = y_train.iloc[valid_idx]
        row_id_valid = row_id_train.iloc[valid_idx]

        fold_model = clone(pipeline)
        fold_model.fit(X_fit, y_fit)
        valid_prob = fold_model.predict_proba(X_valid)[:, 1]

        fold_metrics = compute_binary_metrics(y_valid, valid_prob)
        fold_metrics.update(
            {
                "model_name": model_name,
                "fold": fold_number,
                "validation_rows": int(len(X_valid)),
                "validation_fraud_count": int(y_valid.sum()),
            }
        )
        fold_rows.append(fold_metrics)

        oof_rows.append(
            pd.DataFrame(
                {
                    "transaction_id": row_id_valid.to_numpy(),
                    "y_true": y_valid.to_numpy(),
                    "model_name": model_name,
                    "fold": fold_number,
                    "predicted_probability": valid_prob,
                    "predicted_label_0_5": (valid_prob >= 0.5).astype(int),
                }
            )
        )

    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_predictions_df = pd.concat(oof_rows, ignore_index=True).sort_values(["fold", "transaction_id"]).reset_index(drop=True)

    summary_df = pd.DataFrame(
        [
            {
                "model_name": model_name,
                "metric_scope": "cv_mean",
                "precision": fold_metrics_df["precision"].mean(),
                "recall": fold_metrics_df["recall"].mean(),
                "f1": fold_metrics_df["f1"].mean(),
                "roc_auc": fold_metrics_df["roc_auc"].mean(),
                "pr_auc": fold_metrics_df["pr_auc"].mean(),
            },
            {
                "model_name": model_name,
                "metric_scope": "cv_std",
                "precision": fold_metrics_df["precision"].std(ddof=0),
                "recall": fold_metrics_df["recall"].std(ddof=0),
                "f1": fold_metrics_df["f1"].std(ddof=0),
                "roc_auc": fold_metrics_df["roc_auc"].std(ddof=0),
                "pr_auc": fold_metrics_df["pr_auc"].std(ddof=0),
            },
        ]
    )

    return fold_metrics_df, oof_predictions_df, summary_df


## 7. Baseline Model 1: Logistic Regression

This is the linear baseline. `class_weight='balanced'` helps the model pay more attention to the rare fraud class during fitting.


In [ ]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

logreg_fold_metrics, logreg_oof_predictions, logreg_cv_summary = run_training_cv(
    model_name="logistic_regression",
    pipeline=logreg_pipeline,
    X_train=X_train,
    y_train=y_train,
    row_id_train=row_id_train,
    cv_strategy=cv_strategy,
)

logreg_fold_metrics


## 8. Baseline Model 2: Random Forest

This is the nonlinear tree baseline. The configuration stays intentionally moderate because the goal here is baseline comparison, not final tuning.


In [ ]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_fold_metrics, rf_oof_predictions, rf_cv_summary = run_training_cv(
    model_name="random_forest",
    pipeline=random_forest_pipeline,
    X_train=X_train,
    y_train=y_train,
    row_id_train=row_id_train,
    cv_strategy=cv_strategy,
)

rf_fold_metrics


## 9. Cross-Validation Comparison Summary

This summary compares fold-level mean and variability across the two baseline models. The test set is still untouched at this stage.


In [ ]:
cv_summary = pd.concat([logreg_cv_summary, rf_cv_summary], ignore_index=True)
cv_fold_metrics = pd.concat([logreg_fold_metrics, rf_fold_metrics], ignore_index=True)
cv_oof_predictions = pd.concat([logreg_oof_predictions, rf_oof_predictions], ignore_index=True)

cv_summary.to_csv(NOTEBOOK_TABLES_DIR / "cv_summary.csv", index=False)
cv_fold_metrics.to_csv(NOTEBOOK_TABLES_DIR / "cv_fold_metrics.csv", index=False)
cv_oof_predictions.to_csv(NOTEBOOK_TABLES_DIR / "cv_oof_predictions.csv", index=False)

cv_summary


## 10. Fit Final Baseline Models on the Full Training Split

After comparing the models across training folds, each baseline model is retrained on the full training split so notebook `14_model_evaluation` can score them once on the untouched holdout test set.


In [ ]:
final_logreg_pipeline = clone(logreg_pipeline)
final_random_forest_pipeline = clone(random_forest_pipeline)

final_logreg_pipeline.fit(X_train, y_train)
final_random_forest_pipeline.fit(X_train, y_train)

holdout_predictions = pd.DataFrame(
    {
        "transaction_id": row_id_test.to_numpy(),
        "y_true": y_test.to_numpy(),
        "logistic_regression_probability": final_logreg_pipeline.predict_proba(X_test)[:, 1],
        "random_forest_probability": final_random_forest_pipeline.predict_proba(X_test)[:, 1],
    }
).sort_values("transaction_id").reset_index(drop=True)

holdout_predictions["logistic_regression_prediction_0_5"] = (
    holdout_predictions["logistic_regression_probability"] >= 0.5
).astype(int)
holdout_predictions["random_forest_prediction_0_5"] = (
    holdout_predictions["random_forest_probability"] >= 0.5
).astype(int)

holdout_predictions.to_csv(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv", index=False)
holdout_predictions.head()


## 11. Save Training Outputs

Notebook `14_model_evaluation` should not need to retrain these models. This section saves the fitted pipelines, split metadata, and holdout probabilities needed for comparison.


In [ ]:
split_indices = {
    "train_transaction_ids": row_id_train.tolist(),
    "test_transaction_ids": row_id_test.tolist(),
}
(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json").write_text(
    json.dumps(split_indices, indent=2),
    encoding="utf-8",
)

joblib.dump(
    {
        "model": final_logreg_pipeline,
        "model_name": "logistic_regression",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "cv_splits": N_SPLITS,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib",
)

joblib.dump(
    {
        "model": final_random_forest_pipeline,
        "model_name": "random_forest",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "cv_splits": N_SPLITS,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib",
)

artifact_manifest = pd.DataFrame(
    [
        {"artifact_type": "model", "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib")},
        {"artifact_type": "model", "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib")},
        {"artifact_type": "cv_summary", "path": str(NOTEBOOK_TABLES_DIR / "cv_summary.csv")},
        {"artifact_type": "cv_fold_metrics", "path": str(NOTEBOOK_TABLES_DIR / "cv_fold_metrics.csv")},
        {"artifact_type": "cv_oof_predictions", "path": str(NOTEBOOK_TABLES_DIR / "cv_oof_predictions.csv")},
        {"artifact_type": "holdout_test_predictions", "path": str(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv")},
        {"artifact_type": "split_ids", "path": str(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json")},
    ]
)
artifact_manifest.to_csv(NOTEBOOK_TABLES_DIR / "training_artifact_manifest.csv", index=False)
artifact_manifest


## 12. Training Summary

This notebook ends with a compact handoff summary. Detailed comparison belongs in notebook `14_model_evaluation`.


In [ ]:
training_overview = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "validation_strategy": f"{N_SPLITS}-fold StratifiedKFold on training split",
            "final_fit_status": "trained_on_full_training_split",
            "holdout_test_rows": int(len(holdout_predictions)),
        },
        {
            "model_name": "random_forest",
            "validation_strategy": f"{N_SPLITS}-fold StratifiedKFold on training split",
            "final_fit_status": "trained_on_full_training_split",
            "holdout_test_rows": int(len(holdout_predictions)),
        },
    ]
)
training_overview.to_csv(NOTEBOOK_TABLES_DIR / "training_overview.csv", index=False)

training_report = f"""# Model Training Report

## Purpose
- Train the baseline fraud-detection models on the finalized selected-feature dataset.
- Keep the holdout test set untouched during cross-validation.
- Save the trained artifacts and holdout probabilities for notebook `14_model_evaluation`.

## Training Inputs
- Modeling dataset: `{SELECTED_DATA_FILE}`
- Selected feature count: `{len(selected_features)}`
- Total rows: `{len(model_df)}`
- Fraud rate: `{y.mean():.6f}`
- Test size: `{TEST_SIZE}`
- Random state: `{RANDOM_STATE}`
- Cross-validation: `StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})` on the training split only

## Models Trained
- Logistic Regression with median imputation, standard scaling, and `class_weight='balanced'`
- Random Forest with median imputation and `class_weight='balanced'`

## Handoff to Next Notebook
- Read `cv_summary.csv` and `cv_fold_metrics.csv` to compare fold performance.
- Read `baseline_test_predictions.csv` to evaluate both final baseline models once on the untouched holdout test set.
"""

report_path = NOTEBOOK_TABLES_DIR / "model_training_report.md"
report_path.write_text(training_report, encoding="utf-8")

print(f"Training report saved to: {report_path}")
training_overview
